In [0]:
orders_df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("s3://yuvansecond/de_files/orders.csv")    

In [0]:
orders_df.display()

In [0]:
customer_df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("s3://yuvansecond/de_files/customers.csv")   

In [0]:
customer_df.display()

In [0]:
payment_df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("s3://yuvansecond/de_files/payments.csv")  

In [0]:
payment_df.display()


In [0]:
orders_df.createOrReplaceTempView("orders")
customer_df.createOrReplaceTempView("customers")
payment_df.createOrReplaceTempView("payments")

In [0]:
%sql
select * from orders

In [0]:
%sql
SELECT *
FROM orders o
WHERE o.order_id IN (
    SELECT order_id
    FROM orders where o.amount > 0 and o.customer_id is NOT NULL
    GROUP BY order_id
    HAVING COUNT(*) =1 
);


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW orders_clean AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) as rn
    FROM orders
)
WHERE rn = 1
AND customer_id IS NOT NULL
AND amount > 0;

In [0]:
%sql
select * from orders_clean

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW final_data AS
SELECT 
    o.customer_id,
    DATE(o.order_timestamp) as dt,
    COUNT(o.order_id) as total_orders,
    SUM(o.amount) as total_amount,
    COUNT(CASE WHEN p.payment_status = 'success' THEN 1 END) as success_payments
FROM orders_clean o
LEFT JOIN payments p
    ON o.order_id = p.order_id
GROUP BY o.customer_id, DATE(o.order_timestamp);

In [0]:
%sql
CREATE OR REPLACE TABLE processed_orders AS
SELECT * FROM final_data;

In [0]:
%sql
DROP TABLE IF EXISTS processed_orders_s3;


In [0]:
%sql
CREATE TABLE processed_orders_s3 (
    customer_id INT,
    dt DATE,
    total_orders BIGINT,
    total_amount DOUBLE,
    success_payments BIGINT
)
USING PARQUET
LOCATION 's3://yuvansecond/processed/processed_orders/';

In [0]:
%sql
SELECT * FROM processed_orders;

In [0]:
%sql
INSERT OVERWRITE processed_orders_s3
SELECT * FROM processed_orders;